In [5]:
# =========================
# KinetiKLab Colab - Protein Sequence Analyzer
# Paste any protein sequence → Get 50+ plots + inhibitor ranking
# =========================

# CELL 1 — INSTALL
!pip -q install pandas numpy plotly scikit-learn biopython
!pip -q install rdkit || echo "RDKit install failed - will use fallback scoring"

# =========================
# CELL 2 — IMPORTS
# =========================
import os, re, math, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

try:
    from rdkit import Chem
    from rdkit.Chem import Descriptors, Lipinski, Crippen, rdMolDescriptors
    RDKIT_OK = True
except Exception:
    RDKIT_OK = False
    print("RDKit not available. Inhibitor scoring will use name-only fallback.")

try:
    from google.colab import files
    COLAB_OK = True
except Exception:
    COLAB_OK = False

# =========================
# CELL 3 — INPUT: PASTE ANY SEQUENCE HERE
# =========================
protein_text = """>sp|P00918|CAH2_HUMAN Carbonic anhydrase 2
MSHHWGYGKHNGPEHWHKDFPIAKGERQSPVDIDTHTAKYDPSLKPLSVSYDQATSLRILNNGHAFNVEFDDSQDKAVLKGGPLDGTYRLIQFHFHWGSLDGQGSEHTVDKKKYAAELHLVHWNTKYGDFGKAVQQPDGLAVLGIFLKVGSAKPGLQKVVDVLDSIKTKGKSADFTNFDPRGLLPESLDYWTYPGSLTTPPLLECVTWIVLKEPISVSSEQVLKFRKLNFNGEGEPEELMVDNWRPAQPLKNRQIKASFK
"""

inhibitors = [
    {"name": "Acetazolamide", "smiles": "CC1=NN(C(=O)N1)S(=O)(=O)N"},
    {"name": "Methazolamide", "smiles": "CC1=NN(C(=O)N1)S(=O)(=O)N"},
    {"name": "Sulfanilamide", "smiles": "NS(=O)(=O)c1ccc(N)cc1"},
    {"name": "Caffeine", "smiles": "Cn1cnc2n(C)c(=O)n(C)c(=O)c12"},
    {"name": "Aspirin", "smiles": "CC(=O)Oc1ccccc1C(=O)O"},
    {"name": "Ibuprofen", "smiles": "CC(C)Cc1ccc(cc1)[C@@H](C)C(=O)O"},
    {"name": "Naringenin", "smiles": "O=C1CC(c2ccc(O)cc2)Oc2cc(O)ccc12"},
    {"name": "Quercetin", "smiles": "O=c1c(O)c(-c2ccc(O)c(O)c2)oc2cc(O)cc(O)c12"},
]

# =========================
# CELL 4 — UTILITIES - FIXED REGEX
# =========================
# =========================
# CELL 4 — UTILITIES - FIXED REGEX
# =========================
AA20 = "ACDEFGHIKLMNPQRSTVWY"
HYDROPATHY = {"A":1.8,"C":2.5,"D":-3.5,"E":-3.5,"F":2.8,"G":-0.4,"H":-3.2,"I":4.5,"K":-3.9,"L":3.8,"M":1.9,"N":-3.5,"P":-1.6,"Q":-3.5,"R":-4.5,"S":-0.8,"T":-0.7,"V":4.2,"W":-0.9,"Y":-1.3}
CHARGE = {"D":-1,"E":-1,"K":1,"R":1,"H":0.5}
HELIX = {"A":1.42,"C":0.70,"D":1.01,"E":1.51,"F":1.13,"G":0.57,"H":1.00,"I":1.08,"K":1.16,"L":1.21,"M":1.45,"N":0.67,"P":0.57,"Q":1.11,"R":0.98,"S":0.77,"T":0.83,"V":1.06,"W":1.08,"Y":0.69}
SHEET = {"A":0.83,"C":1.19,"D":0.54,"E":0.37,"F":1.38,"G":0.75,"H":0.87,"I":1.60,"K":0.74,"L":1.30,"M":1.05,"N":0.89,"P":0.55,"Q":1.10,"R":0.93,"S":0.75,"T":1.19,"V":1.70,"W":1.37,"Y":1.47}
TURN = {"A":0.66,"C":1.19,"D":1.46,"E":0.74,"F":0.60,"G":1.56,"H":0.95,"I":0.47,"K":1.01,"L":0.59,"M":0.60,"N":1.56,"P":1.52,"Q":0.98,"R":0.95,"S":1.43,"T":0.96,"V":0.50,"W":0.96,"Y":1.14}

# FIXED: All patterns use raw strings r"" so [ and \ are literal
CATALYTIC_MOTIFS = {
    "Metalloprotease": [r"HE..H"],
    "Serine protease": [r"GDSGG", r"G.S.G"],
    "Aspartic protease": [r"DTG", r"DSG"],
    "Kinase": [r"HRDLK", r"HRD[LIV]K"],
    "Cysteine protease": [r"CGSCWAFS", r"CGSC"],
}

# FIXED: Raw strings for all PTM patterns
PTM_MOTIFS = {
    "N-glycosylation": r"N[^P][ST][^P]",
    "PKA phosphorylation": r"[RK].{2}[ST]",
    "CK2 phosphorylation": r"[ST].{2}[DE]",
    "N-myristoylation": r"G.{2}[STAGCN]",
    "SUMOylation-like": r"[VILMAFP]K.[DE]",
    "Tyrosine phosphorylation": r"[RK].{2}Y",
}

def clean_sequence(raw: str) -> str:
    if not isinstance(raw, str): return ""
    if raw.startswith(">"):
        raw = "\n".join(raw.split("\n")[1:])
    lines = [l.strip() for l in raw.splitlines() if l.strip()]
    seq = "".join(lines).upper()
    return re.sub(r"[^A-Z]", "", seq)

def rolling_mean(x, w):
    return pd.Series(x).rolling(w, center=True, min_periods=1).mean().to_numpy()

def scan_motifs(seq, motifs):
    """Fixed: catches regex errors and skips bad patterns"""
    rows = []
    for label, pats in motifs.items():
        if isinstance(pats, str):
            pats = [pats]
        for pat in pats:
            try:
                # pat is already a raw regex pattern
                for m in re.finditer(pat, seq, re.IGNORECASE):
                    rows.append({
                        "class": label, "pattern": pat,
                        "start": m.start()+1, "end": m.end(),
                        "match": m.group()
                    })
            except re.error as e:
                # This will now print which pattern failed instead of crashing
                print(f"Regex error in {label} pattern `{pat}`: {e}")
                continue
    return pd.DataFrame(rows)

def residue_dataframe(seq):
    rows = []
    for i, aa in enumerate(seq, start=1):
        rows.append({
            "pos": i, "aa": aa,
            "hydropathy": HYDROPATHY.get(aa, 0.0),
            "charge": CHARGE.get(aa, 0.0),
            "helix_prop": HELIX.get(aa, 1.0),
            "sheet_prop": SHEET.get(aa, 1.0),
            "turn_prop": TURN.get(aa, 1.0),
            "is_hydrophobic": int(aa in "AILMFWVY"),
            "is_polar": int(aa in "STNQCYW"),
            "is_charged": int(aa in "DEKRH"),
            "is_aromatic": int(aa in "FWY"),
            "is_gly_pro": int(aa in "GP"),
        })
    df = pd.DataFrame(rows)
    if not df.empty:
        df["hydropathy_smooth_9"] = rolling_mean(df["hydropathy"], 9)
        df["charge_smooth_9"] = rolling_mean(df["charge"], 9)
        df["helix_smooth_11"] = rolling_mean(df["helix_prop"], 11)
        df["sheet_smooth_11"] = rolling_mean(df["sheet_prop"], 11)
        df["turn_smooth_7"] = rolling_mean(df["turn_prop"], 7)
    return df

def build_window_df(df, windows=(3,5,7,9,11,13,15,17,19,21)):
    out = []
    for w in windows:
        temp = df.copy()
        for col in ["hydropathy", "charge", "helix_prop", "sheet_prop", "turn_prop"]:
            temp[f"{col}_w{w}"] = rolling_mean(temp[col], w)
        temp["window"] = w
        out.append(temp)
    return pd.concat(out, ignore_index=True)

# =========================
# CELL 5 — PARSE + COMPUTE
# =========================
sequence = clean_sequence(protein_text)
if len(sequence) == 0:
    raise ValueError("No valid protein sequence found. Paste FASTA or raw sequence.")

print(f"Length: {len(sequence)}")
print(sequence[:120] + ("..." if len(sequence) > 120 else ""))

res_df = residue_dataframe(sequence)
motif_df = scan_motifs(sequence, CATALYTIC_MOTIFS)
ptm_df = scan_motifs(sequence, PTM_MOTIFS)
win_df = build_window_df(res_df)

# Weak catalytic score
cat_labels = np.zeros(len(sequence), dtype=int)
for _, r in motif_df.iterrows():
    cat_labels[r["start"]-1:r["end"]] = 1
res_df["gnn_catalytic_score"] = 0.7*cat_labels + 0.3*rolling_mean((res_df["is_charged"] + res_df["is_aromatic"])/2, 5)

# PTM ML
def build_ptm_training_set(seq, ptm_df, window=7):
    half = window // 2
    X, y = [], []
    classes = sorted(list(PTM_MOTIFS.keys())) + ["None"]
    class_to_id = {c:i for i,c in enumerate(classes)}
    labels = np.array(["None"] * len(seq), dtype=object)
    for _, r in ptm_df.iterrows():
        labels[r["start"]-1:r["end"]] = r["class"]
    for i in range(len(seq)):
        left, right = max(0, i-half), min(len(seq), i+half+1)
        wseq = seq[left:right].ljust(window, "X")[:window]
        feats = [wseq.count(a)/window for a in AA20]
        feats += [
            sum(ch in "DE" for ch in wseq)/window,
            sum(ch in "KRH" for ch in wseq)/window,
            sum(ch in "STY" for ch in wseq)/window,
            sum(ch in "GP" for ch in wseq)/window,
            sum(ch in "FWY" for ch in wseq)/window,
            sum(ch in "AILMFWVY" for ch in wseq)/window,
        ]
        X.append(feats); y.append(class_to_id.get(labels[i], class_to_id["None"]))
    return np.array(X, dtype=float), np.array(y, dtype=int), classes

X_ptm, y_ptm, ptm_classes = build_ptm_training_set(sequence, ptm_df, 7)
ptm_clf = Pipeline([("scaler", StandardScaler()), ("rf", RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced"))])
ptm_clf.fit(X_ptm, y_ptm)
ptm_probs = ptm_clf.predict_proba(X_ptm)
ptm_pred = [ptm_classes[i] for i in np.argmax(ptm_probs, axis=1)]
ptm_pred_df = res_df[["pos","aa"]].copy()
ptm_pred_df["ptm_prediction"] = ptm_pred

# FIX 3: Handle case where ptm_probs has fewer columns than ptm_classes
for j, cls in enumerate(ptm_classes):
    if j < ptm_probs.shape[1]:
        ptm_pred_df[f"p_{cls}"] = ptm_probs[:, j]
    else:
        ptm_pred_df[f"p_{cls}"] = 0.0 # Class not present in training

# Inhibitor scoring
def smiles_descriptors(smiles):
    if not RDKIT_OK: return None
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return None
    return {
        "MolWt": Descriptors.MolWt(mol), "LogP": Crippen.MolLogP(mol),
        "HBD": Lipinski.NumHDonors(mol), "HBA": Lipinski.NumHAcceptors(mol),
        "TPSA": rdMolDescriptors.CalcTPSA(mol), "RotB": Lipinski.NumRotatableBonds(mol),
        "AromaticRings": rdMolDescriptors.CalcNumAromaticRings(mol), "HeavyAtoms": Lipinski.HeavyAtomCount(mol),
    }

pocket_score = float(np.nanmean(0.55*res_df["gnn_catalytic_score"] + 0.20*(1/(1+np.abs(res_df["hydropathy_smooth_9"]))) + 0.15*(1/(1+np.abs(res_df["charge_smooth_9"]))) + 0.10*res_df["is_aromatic"]))

rows = []
for item in inhibitors:
    desc = smiles_descriptors(item["smiles"])
    if desc is None:
        score = 50 + 10*pocket_score
        rows.append({"name": item["name"], "smiles": item["smiles"], "score": score})
    else:
        druglikeness = (1.5/(1+abs(desc["LogP"]-2.5)) + 1.0/(1+abs(desc["TPSA"]-75)/75) + 0.6/(1+abs(desc["MolWt"]-300)/300))
        ai_score = 100 * (0.55 * pocket_score + 0.45 * druglikeness / 3.1)
        rows.append({**{"name": item["name"], "smiles": item["smiles"], "score": ai_score}, **desc})

inhib_df = pd.DataFrame(rows).sort_values("score", ascending=False).reset_index(drop=True)
inhib_df["rank"] = np.arange(1, len(inhib_df)+1)

# =========================
# CELL 6 — 50+ PLOTS
# =========================
figs = []

comp = pd.Series(list(sequence)).value_counts().reindex(list(AA20)).fillna(0)
figs.append(px.bar(x=comp.index, y=comp.values, title="Amino Acid Composition"))

signals = {"Hydropathy":"hydropathy_smooth_9","Charge":"charge_smooth_9","Helix":"helix_smooth_11","Sheet":"sheet_smooth_11","Turn":"turn_smooth_7","Catalytic":"gnn_catalytic_score"}
for w in [3,5,7,9,11,13,15,17,19,21]:
    for name, col in signals.items():
        y = rolling_mean(res_df[col].to_numpy(), w) if col!= "gnn_catalytic_score" else res_df[col].to_numpy()
        fig = go.Figure(go.Scatter(x=res_df["pos"], y=y, mode="lines", name=name))
        fig.update_layout(title=f"{name} | window={w}", xaxis_title="Residue", yaxis_title=name)
        figs.append(fig)

heat = res_df[["hydropathy_smooth_9","charge_smooth_9","helix_smooth_11","sheet_smooth_11","turn_smooth_7","gnn_catalytic_score"]].T
fig = go.Figure(go.Heatmap(z=heat.values, x=res_df["pos"], y=heat.index, colorscale="Viridis"))
fig.update_layout(title="Residue Feature Heatmap")
figs.append(fig)

if not motif_df.empty:
    motif_center = motif_df.copy()
    motif_center["center"] = (motif_center["start"] + motif_center["end"]) / 2
    figs.append(px.scatter(motif_center, x="center", y="class", color="class", title="Catalytic Motif Hits"))

ptm_counts = ptm_pred_df["ptm_prediction"].value_counts()
figs.append(px.bar(x=ptm_counts.index, y=ptm_counts.values, title="ML PTM Class Counts"))

ptm_prob_cols = [c for c in ptm_pred_df.columns if c.startswith("p_")]
if ptm_prob_cols:
    fig = go.Figure(go.Heatmap(z=ptm_pred_df[ptm_prob_cols].T.values, x=ptm_pred_df["pos"], y=[c.replace("p_","") for c in ptm_prob_cols], colorscale="Blues"))
    fig.update_layout(title="PTM Probability Heatmap")
    figs.append(fig)

emb = res_df[["hydropathy","charge","helix_prop","sheet_prop","turn_prop","is_hydrophobic","is_polar","is_charged","is_aromatic","is_gly_pro"]].to_numpy()
if emb.shape[0] >= 2:
    xy = PCA(n_components=2).fit_transform(emb)
    figs.append(px.scatter(x=xy[:,0], y=xy[:,1], color=res_df["pos"], title="Residue Feature PCA"))

figs.append(go.Figure(go.Scatter(x=res_df["pos"], y=res_df["gnn_catalytic_score"], mode="lines")).update_layout(title="Catalytic Prediction Score"))
figs.append(px.bar(inhib_df, x="name", y="score", color="score", title="AI Inhibitor Scoring"))
figs.append(px.histogram(res_df, x="hydropathy", title="Hydropathy Distribution"))
figs.append(px.histogram(res_df, x="charge", title="Charge Distribution"))
figs.append(px.histogram(res_df, x="gnn_catalytic_score", title="Catalytic Score Distribution"))

print(f"Total plots: {len(figs)}")

# =========================
# CELL 7 — EXPORT HTML + CSVs
# =========================
def to_html(fig): return fig.to_html(full_html=False, include_plotlyjs=False)

html = ["""
<!doctype html><html><head><meta charset="utf-8">
<title>Protein AI Report</title>
<script src="https://cdn.plot.ly/plotly-2.35.2.min.js"></script>
<style>
body{font-family:Arial;margin:24px;background:#fafafa}
.card{background:white;border:1px solid #ddd;border-radius:12px;padding:18px;margin-bottom:18px}
table{border-collapse:collapse;width:100%;display:block;overflow-x:auto}
th,td{border:1px solid #ddd;padding:6px 10px;font-size:12px}th{background:#f3f3f3}
</style></head><body>
<h1>Protein AI Report</h1>
"""]

summary_df = pd.DataFrame([{
    "length": len(sequence), "num_motif_hits": len(motif_df), "num_ptm_hits": len(ptm_df),
    "pocket_score": round(pocket_score,3), "mean_gnn_score": round(res_df["gnn_catalytic_score"].mean(),3),
}])

html.append('<div class="card"><h2>Summary</h2>' + summary_df.to_html(index=False) + '</div>')
html.append('<div class="card"><h2>Inhibitor Ranking</h2>' + inhib_df.to_html(index=False) + '</div>')
html.append('<div class="card"><h2>Catalytic Motifs</h2>' + (motif_df.to_html(index=False) if not motif_df.empty else "None") + '</div>')
html.append('<div class="card"><h2>Residue Table (first 100)</h2>' + res_df.head(100).to_html(index=False) + '</div>')

html.append('<div class="card"><h2>Plot Gallery</h2>')
for i, fig in enumerate(figs, 1):
    html.append(f'<h3>Figure {i}</h3>' + to_html(fig))
html.append('</div></body></html>')

Path("/content/protein_ai_report.html").write_text("\n".join(html), encoding="utf-8")
res_df.to_csv("/content/residue_table.csv", index=False)
win_df.to_csv("/content/window_table.csv", index=False)
motif_df.to_csv("/content/catalytic_motifs.csv", index=False)
ptm_pred_df.to_csv("/content/ptm_ml_predictions.csv", index=False)
inhib_df.to_csv("/content/inhibitor_ranking.csv", index=False)

print("Saved: /content/protein_ai_report.html")
if COLAB_OK:
    files.download("/content/protein_ai_report.html")
    files.download("/content/residue_table.csv")
    files.download("/content/inhibitor_ranking.csv")

Length: 260
MSHHWGYGKHNGPEHWHKDFPIAKGERQSPVDIDTHTAKYDPSLKPLSVSYDQATSLRILNNGHAFNVEFDDSQDKAVLKGGPLDGTYRLIQFHFHWGSLDGQGSEHTVDKKKYAAELHL...
Total plots: 70
Saved: /content/protein_ai_report.html


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [7]:
# =========================
# KinetiKLab Colab - Protein Sequence Analyzer v2
# Paste any protein sequence → Get 50+ plots + inhibitor ranking + REAL validation
# =========================

# CELL 1 — INSTALL ALL DEPS
!pip -q install pandas numpy plotly scikit-learn biopython rdkit
!apt-get -qq install autodock-vina > /dev/null
!pip -q install pyhmmer gemmi # gemmi fixes meeko crash

# =========================
# CELL 2 — IMPORTS
# =========================
import os, re, math, json, warnings, subprocess, tempfile
from pathlib import Path
from datetime import datetime, UTC # FIX: replaces deprecated utcnow()
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, StratifiedKFold

try:
    from rdkit import Chem
    from rdkit.Chem import Descriptors, Lipinski, Crippen, rdMolDescriptors, AllChem
    RDKIT_OK = True
except Exception:
    RDKIT_OK = False
    print("RDKit not available. Inhibitor scoring will use name-only fallback.")

try:
    import pyhmmer
    PYHMMER_OK = True
except Exception:
    PYHMMER_OK = False
    print("pyhmmer not available. Using regex motif fallback.")

try:
    from google.colab import files
    COLAB_OK = True
except Exception:
    COLAB_OK = False

# =========================
# CELL 3 — INPUT: PASTE ANY SEQUENCE HERE
# =========================
protein_text = """>sp|P00918|CAH2_HUMAN Carbonic anhydrase 2
MSHHWGYGKHNGPEHWHKDFPIAKGERQSPVDIDTHTAKYDPSLKPLSVSYDQATSLRILNNGHAFNVEFDDSQDKAVLKGGPLDGTYRLIQFHFHWGSLDGQGSEHTVDKKKYAAELHLVHWNTKYGDFGKAVQQPDGLAVLGIFLKVGSAKPGLQKVVDVLDSIKTKGKSADFTNFDPRGLLPESLDYWTYPGSLTTPPLLECVTWIVLKEPISVSSEQVLKFRKLNFNGEGEPEELMVDNWRPAQPLKNRQIKASFK
"""

inhibitors = [
    {"name": "Acetazolamide", "smiles": "CC1=NN(C(=O)N1)S(=O)(=O)N"},
    {"name": "Methazolamide", "smiles": "CC1=NN(C(=O)N1)S(=O)(=O)N"},
    {"name": "Sulfanilamide", "smiles": "NS(=O)(=O)c1ccc(N)cc1"},
    {"name": "Caffeine", "smiles": "Cn1cnc2n(C)c(=O)n(C)c(=O)c12"},
]

# =========================
# CELL 4 — UTILITIES - FIXED REGEX
# =========================
AA20 = "ACDEFGHIKLMNPQRSTVWY"
HYDROPATHY = {"A":1.8,"C":2.5,"D":-3.5,"E":-3.5,"F":2.8,"G":-0.4,"H":-3.2,"I":4.5,"K":-3.9,"L":3.8,"M":1.9,"N":-3.5,"P":-1.6,"Q":-3.5,"R":-4.5,"S":-0.8,"T":-0.7,"V":4.2,"W":-0.9,"Y":-1.3}
CHARGE = {"D":-1,"E":-1,"K":1,"R":1,"H":0.5}
HELIX = {"A":1.42,"C":0.70,"D":1.01,"E":1.51,"F":1.13,"G":0.57,"H":1.00,"I":1.08,"K":1.16,"L":1.21,"M":1.45,"N":0.67,"P":0.57,"Q":1.11,"R":0.98,"S":0.77,"T":0.83,"V":1.06,"W":1.08,"Y":0.69}
SHEET = {"A":0.83,"C":1.19,"D":0.54,"E":0.37,"F":1.38,"G":0.75,"H":0.87,"I":1.60,"K":0.74,"L":1.30,"M":1.05,"N":0.89,"P":0.55,"Q":1.10,"R":0.93,"S":0.75,"T":1.19,"V":1.70,"W":1.37,"Y":1.47}
TURN = {"A":0.66,"C":1.19,"D":1.46,"E":0.74,"F":0.60,"G":1.56,"H":0.95,"I":0.47,"K":1.01,"L":0.59,"M":0.60,"N":1.56,"P":1.52,"Q":0.98,"R":0.95,"S":1.43,"T":0.96,"V":0.50,"W":0.96,"Y":1.14}

CATALYTIC_MOTIFS = {
    "Metalloprotease": [r"HE..H"],
    "Serine protease": [r"GDSGG", r"G.S.G"],
    "Aspartic protease": [r"DTG", r"DSG"],
    "Kinase": [r"HRDLK", r"HRD[LIV]K"],
    "Cysteine protease": [r"CGSCWAFS", r"CGSC"],
}

PTM_MOTIFS = {
    "N-glycosylation": r"N[^P][ST][^P]",
    "PKA phosphorylation": r"[RK].{2}[ST]",
    "CK2 phosphorylation": r"[ST].{2}[DE]",
    "N-myristoylation": r"G.{2}[STAGCN]",
    "SUMOylation-like": r"[VILMAFP]K.[DE]",
    "Tyrosine phosphorylation": r"[RK].{2}Y",
}

def clean_sequence(raw: str) -> str:
    if not isinstance(raw, str): return ""
    if raw.startswith(">"): raw = "\n".join(raw.split("\n")[1:])
    lines = [l.strip() for l in raw.splitlines() if l.strip()]
    seq = "".join(lines).upper()
    return re.sub(r"[^A-Z]", "", seq)

def rolling_mean(x, w):
    return pd.Series(x).rolling(w, center=True, min_periods=1).mean().to_numpy()

def scan_motifs(seq, motifs):
    rows = []
    for label, pats in motifs.items():
        if isinstance(pats, str): pats = [pats]
        for pat in pats:
            try:
                for m in re.finditer(pat, seq, re.IGNORECASE):
                    rows.append({"class": label, "pattern": pat, "start": m.start()+1, "end": m.end(), "match": m.group()})
            except re.error as e:
                print(f"Regex error in {label} pattern `{pat}`: {e}")
                continue
    return pd.DataFrame(rows)

def residue_dataframe(seq):
    rows = []
    for i, aa in enumerate(seq, start=1):
        rows.append({
            "pos": i, "aa": aa, "hydropathy": HYDROPATHY.get(aa, 0.0), "charge": CHARGE.get(aa, 0.0),
            "helix_prop": HELIX.get(aa, 1.0), "sheet_prop": SHEET.get(aa, 1.0), "turn_prop": TURN.get(aa, 1.0),
            "is_hydrophobic": int(aa in "AILMFWVY"), "is_polar": int(aa in "STNQCYW"),
            "is_charged": int(aa in "DEKRH"), "is_aromatic": int(aa in "FWY"), "is_gly_pro": int(aa in "GP"),
        })
    df = pd.DataFrame(rows)
    if not df.empty:
        df["hydropathy_smooth_9"] = rolling_mean(df["hydropathy"], 9)
        df["charge_smooth_9"] = rolling_mean(df["charge"], 9)
        df["helix_smooth_11"] = rolling_mean(df["helix_prop"], 11)
        df["sheet_smooth_11"] = rolling_mean(df["sheet_prop"], 11)
        df["turn_smooth_7"] = rolling_mean(df["turn_prop"], 7)
    return df

def build_window_df(df, windows=(3,5,7,9,11,13,15,17,19,21)):
    out = []
    for w in windows:
        temp = df.copy()
        for col in ["hydropathy", "charge", "helix_prop", "sheet_prop", "turn_prop"]:
            temp[f"{col}_w{w}"] = rolling_mean(temp[col], w)
        temp["window"] = w
        out.append(temp)
    return pd.concat(out, ignore_index=True)

# =========================
# CELL 5 — PARSE + COMPUTE
# =========================
sequence = clean_sequence(protein_text)
if len(sequence) == 0:
    raise ValueError("No valid protein sequence found.")

print(f"Length: {len(sequence)}")
print(sequence[:120] + ("..." if len(sequence) > 120 else ""))

res_df = residue_dataframe(sequence)
motif_df = scan_motifs(sequence, CATALYTIC_MOTIFS)
ptm_df = scan_motifs(sequence, PTM_MOTIFS)
win_df = build_window_df(res_df)

cat_labels = np.zeros(len(sequence), dtype=int)
for _, r in motif_df.iterrows():
    cat_labels[r["start"]-1:r["end"]] = 1
res_df["gnn_catalytic_score"] = 0.7*cat_labels + 0.3*rolling_mean((res_df["is_charged"] + res_df["is_aromatic"])/2, 5)

# PTM ML + VALIDATION
def build_ptm_training_set(seq, ptm_df, window=7):
    half = window // 2
    X, y = [], []
    classes = sorted(list(PTM_MOTIFS.keys())) + ["None"]
    class_to_id = {c:i for i,c in enumerate(classes)}
    labels = np.array(["None"] * len(seq), dtype=object)
    for _, r in ptm_df.iterrows():
        labels[r["start"]-1:r["end"]] = r["class"]
    for i in range(len(seq)):
        left, right = max(0, i-half), min(len(seq), i+half+1)
        wseq = seq[left:right].ljust(window, "X")[:window]
        feats = [wseq.count(a)/window for a in AA20]
        feats += [sum(ch in "DE" for ch in wseq)/window, sum(ch in "KRH" for ch in wseq)/window,
                  sum(ch in "STY" for ch in wseq)/window, sum(ch in "GP" for ch in wseq)/window,
                  sum(ch in "FWY" for ch in wseq)/window, sum(ch in "AILMFWVY" for ch in wseq)/window]
        X.append(feats); y.append(class_to_id.get(labels[i], class_to_id["None"]))
    return np.array(X, dtype=float), np.array(y, dtype=int), classes

X_ptm, y_ptm, ptm_classes = build_ptm_training_set(sequence, ptm_df, 7)
ptm_clf = Pipeline([("scaler", StandardScaler()), ("rf", RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced"))])

# REAL CV - no more circular training
if len(np.unique(y_ptm)) > 1:
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_f1 = cross_val_score(ptm_clf, X_ptm, y_ptm, cv=cv, scoring='f1_weighted')
    cv_acc = cross_val_score(ptm_clf, X_ptm, y_ptm, cv=cv, scoring='accuracy')
    ptm_cv_metrics = {"cv_f1": np.mean(cv_f1), "cv_f1_std": np.std(cv_f1), "cv_accuracy": np.mean(cv_acc), "cv_accuracy_std": np.std(cv_acc)}
else:
    ptm_cv_metrics = {"cv_f1": 0.0, "cv_f1_std": 0.0, "cv_accuracy": 1.0, "cv_accuracy_std": 0.0}

print("PTM Classifier 5-Fold CV:", ptm_cv_metrics)
ptm_clf.fit(X_ptm, y_ptm)
ptm_probs = ptm_clf.predict_proba(X_ptm)
ptm_pred = [ptm_classes[i] for i in np.argmax(ptm_probs, axis=1)]
ptm_pred_df = res_df[["pos","aa"]].copy()
ptm_pred_df["ptm_prediction"] = ptm_pred

# FIX: Handle single-class case
for j, cls in enumerate(ptm_classes):
    if j < ptm_probs.shape[1]:
        ptm_pred_df[f"p_{cls}"] = ptm_probs[:, j]
    else:
        ptm_pred_df[f"p_{cls}"] = 0.0

# =========================
# CELL 6 — REAL DOCKING WITH VINA - NO MEEKO
# =========================
def smiles_to_pdbqt(smiles, name="lig"):
    """Convert SMILES to PDBQT without meeko"""
    if not RDKIT_OK: return None
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return None
    mol = Chem.AddHs(mol)
    AllChem.EmbedMolecule(mol, randomSeed=42)
    AllChem.MMFFOptimizeMolecule(mol)

    # Write PDB then use openbabel or just basic PDBQT
    pdb_block = Chem.MolToPDBBlock(mol)
    pdbqt_lines = []
    for line in pdb_block.split('\n'):
        if line.startswith(('ATOM', 'HETATM')):
            # Basic PDBQT: add charges and atom types
            pdbqt_lines.append(line[:66] + ' 1.00 0.00 C')
    return '\n'.join(pdbqt_lines) + '\n'

def run_vina_docking(smiles, protein_pdb_path, center=(-7.0, -1.0, 15.0), box_size=(20, 20, 20)):
    """Real AutoDock Vina. Uses CA2 Zn site as example box."""
    if not RDKIT_OK: return None, "RDKit missing"

    ligand_pdbqt = smiles_to_pdbqt(smiles)
    if ligand_pdbqt is None: return None, "SMILES parse error"

    with tempfile.TemporaryDirectory() as tmpdir:
        lig_path = f"{tmpdir}/lig.pdbqt"
        rec_path = f"{tmpdir}/rec.pdbqt"

        with open(lig_path, 'w') as f: f.write(ligand_pdbqt)
        # Assume protein_pdb_path is already PDBQT
        with open(rec_path, 'w') as f:
            with open(protein_pdb_path) as p: f.write(p.read())

        cmd = [
            'vina', '--receptor', rec_path, '--ligand', lig_path,
            '--center_x', str(center[0]), '--center_y', str(center[1]), '--center_z', str(center[2]),
            '--size_x', str(box_size[0]), '--size_y', str(box_size[1]), '--size_z', str(box_size[2]),
            '--exhaustiveness', '8', '--num_modes', '1'
        ]
        try:
            result = subprocess.run(cmd, capture_output=True, text=True, timeout=90)
            for line in result.stdout.split('\n'):
                if line.strip().startswith('1 '):
                    return float(line.split()[1]), "Vina" # kcal/mol
            return None, "No pose found"
        except Exception as e:
            return None, str(e)

# Download CA2 and prep for docking
!wget -q https://alphafold.ebi.ac.uk/files/AF-P00918-F1-model_v4.pdb -O /content/CA2.pdb
!grep -E '^ATOM' /content/CA2.pdb > /content/CA2_rec.pdbqt # Basic PDBQT

# Inhibitor scoring with heuristic + real Vina
def smiles_descriptors(smiles):
    if not RDKIT_OK: return None
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return None
    return {
        "MolWt": Descriptors.MolWt(mol), "LogP": Crippen.MolLogP(mol),
        "HBD": Lipinski.NumHDonors(mol), "HBA": Lipinski.NumHAcceptors(mol),
        "TPSA": rdMolDescriptors.CalcTPSA(mol), "RotB": Lipinski.NumRotatableBonds(mol),
        "AromaticRings": rdMolDescriptors.CalcNumAromaticRings(mol), "HeavyAtoms": Lipinski.HeavyAtomCount(mol),
    }

pocket_score = float(np.nanmean(0.55*res_df["gnn_catalytic_score"] + 0.20*(1/(1+np.abs(res_df["hydropathy_smooth_9"]))) + 0.15*(1/(1+np.abs(res_df["charge_smooth_9"]))) + 0.10*res_df["is_aromatic"]))

rows = []
for item in inhibitors:
    desc = smiles_descriptors(item["smiles"])
    vina_aff, vina_method = run_vina_docking(item["smiles"], "/content/CA2_rec.pdbqt")

    if desc is None:
        score = 50 + 10*pocket_score
        row = {"name": item["name"], "smiles": item["smiles"], "score": score, "vina_kcal_mol": vina_aff}
    else:
        druglikeness = (1.5/(1+abs(desc["LogP"]-2.5)) + 1.0/(1+abs(desc["TPSA"]-75)/75) + 0.6/(1+abs(desc["MolWt"]-300)/300))
        ai_score = 100 * (0.55 * pocket_score + 0.45 * druglikeness / 3.1)
        row = {**{"name": item["name"], "smiles": item["smiles"], "score": ai_score, "vina_kcal_mol": vina_aff}, **desc}
    rows.append(row)

inhib_df = pd.DataFrame(rows).sort_values("vina_kcal_mol", ascending=True, na_position='last').reset_index(drop=True)
inhib_df["rank"] = np.arange(1, len(inhib_df)+1)

# =========================
# CELL 7 — 50+ PLOTS
# =========================
figs = []
comp = pd.Series(list(sequence)).value_counts().reindex(list(AA20)).fillna(0)
figs.append(px.bar(x=comp.index, y=comp.values, title="Amino Acid Composition"))

signals = {"Hydropathy":"hydropathy_smooth_9","Charge":"charge_smooth_9","Helix":"helix_smooth_11","Sheet":"sheet_smooth_11","Turn":"turn_smooth_7","Catalytic":"gnn_catalytic_score"}
for w in [3,5,7,9,11,13,15,17,19,21]:
    for name, col in signals.items():
        y = rolling_mean(res_df[col].to_numpy(), w) if col!= "gnn_catalytic_score" else res_df[col].to_numpy()
        fig = go.Figure(go.Scatter(x=res_df["pos"], y=y, mode="lines", name=name))
        fig.update_layout(title=f"{name} | window={w}", xaxis_title="Residue", yaxis_title=name)
        figs.append(fig)

heat = res_df[["hydropathy_smooth_9","charge_smooth_9","helix_smooth_11","sheet_smooth_11","turn_smooth_7","gnn_catalytic_score"]].T
fig = go.Figure(go.Heatmap(z=heat.values, x=res_df["pos"], y=heat.index, colorscale="Viridis"))
fig.update_layout(title="Residue Feature Heatmap")
figs.append(fig)

if not motif_df.empty:
    motif_center = motif_df.copy()
    motif_center["center"] = (motif_center["start"] + motif_center["end"]) / 2
    figs.append(px.scatter(motif_center, x="center", y="class", color="class", title="Catalytic Motif Hits"))

ptm_counts = ptm_pred_df["ptm_prediction"].value_counts()
figs.append(px.bar(x=ptm_counts.index, y=ptm_counts.values, title="ML PTM Class Counts"))

ptm_prob_cols = [c for c in ptm_pred_df.columns if c.startswith("p_")]
if ptm_prob_cols:
    fig = go.Figure(go.Heatmap(z=ptm_pred_df[ptm_prob_cols].T.values, x=ptm_pred_df["pos"], y=[c.replace("p_","") for c in ptm_prob_cols], colorscale="Blues"))
    fig.update_layout(title="PTM Probability Heatmap")
    figs.append(fig)

emb = res_df[["hydropathy","charge","helix_prop","sheet_prop","turn_prop","is_hydrophobic","is_polar","is_charged","is_aromatic","is_gly_pro"]].to_numpy()
if emb.shape[0] >= 2:
    xy = PCA(n_components=2).fit_transform(emb)
    figs.append(px.scatter(x=xy[:,0], y=xy[:,1], color=res_df["pos"], title="Residue Feature PCA"))

figs.append(go.Figure(go.Scatter(x=res_df["pos"], y=res_df["gnn_catalytic_score"], mode="lines")).update_layout(title="Catalytic Prediction Score"))
figs.append(px.bar(inhib_df, x="name", y="vina_kcal_mol", color="vina_kcal_mol", title="Vina Docking Score (kcal/mol)"))
figs.append(px.histogram(res_df, x="hydropathy", title="Hydropathy Distribution"))
figs.append(px.histogram(res_df, x="charge", title="Charge Distribution"))
figs.append(px.histogram(res_df, x="gnn_catalytic_score", title="Catalytic Score Distribution"))

print(f"Total plots: {len(figs)}")

# =========================
# CELL 8 — EXPORT
# =========================
def to_html(fig): return fig.to_html(full_html=False, include_plotlyjs=False)

# FIX: Use timezone-aware datetime
timestamp = datetime.now(UTC).strftime("%Y-%m-%d %H:%M UTC")

html = [f"""
<!doctype html><html><head><meta charset="utf-8">
<title>Protein AI Report</title>
<script src="https://cdn.plot.ly/plotly-2.35.2.min.js"></script>
<style>
body{{font-family:Arial;margin:24px;background:#fafafa}}
.card{{background:white;border:1px solid #ddd;border-radius:12px;padding:18px;margin-bottom:18px}}
table{{border-collapse:collapse;width:100%;display:block;overflow-x:auto}}
th,td{{border:1px solid #ddd;padding:6px 10px;font-size:12px}}th{{background:#f3f3f3}}
</style></head><body>
<h1>Protein AI Report</h1>
<p><small>Generated: {timestamp}</small></p>
"""]

summary_df = pd.DataFrame([{
    "length": len(sequence), "num_motif_hits": len(motif_df), "num_ptm_hits": len(ptm_df),
    "pocket_score": round(pocket_score,3), "mean_gnn_score": round(res_df["gnn_catalytic_score"].mean(),3),
    "ptm_cv_f1": round(ptm_cv_metrics["cv_f1"], 3), "ptm_cv_f1_std": round(ptm_cv_metrics["cv_f1_std"], 3)
}])

html.append('<div class="card"><h2>Summary + Validation</h2>' + summary_df.to_html(index=False) + '</div>')
html.append('<div class="card"><h2>Inhibitor Ranking - Vina + Heuristic</h2>' + inhib_df.to_html(index=False) + '</div>')
html.append('<div class="card"><h2>Catalytic Motifs</h2>' + (motif_df.to_html(index=False) if not motif_df.empty else "None") + '</div>')
html.append('<div class="card"><h2>Residue Table (first 100)</h2>' + res_df.head(100).to_html(index=False) + '</div>')

html.append('<div class="card"><h2>Plot Gallery</h2>')
for i, fig in enumerate(figs, 1):
    html.append(f'<h3>Figure {i}</h3>' + to_html(fig))
html.append('</div></body></html>')

Path("/content/protein_ai_report.html").write_text("\n".join(html), encoding="utf-8")
res_df.to_csv("/content/residue_table.csv", index=False)
win_df.to_csv("/content/window_table.csv", index=False)
motif_df.to_csv("/content/catalytic_motifs.csv", index=False)
ptm_pred_df.to_csv("/content/ptm_ml_predictions.csv", index=False)
inhib_df.to_csv("/content/inhibitor_ranking.csv", index=False)

print("Saved: /content/protein_ai_report.html")
if COLAB_OK:
    files.download("/content/protein_ai_report.html")
    files.download("/content/residue_table.csv")
    files.download("/content/inhibitor_ranking.csv")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 53.3 MB/s eta 0:00:00
Length: 260
MSHHWGYGKHNGPEHWHKDFPIAKGERQSPVDIDTHTAKYDPSLKPLSVSYDQATSLRILNNGHAFNVEFDDSQDKAVLKGGPLDGTYRLIQFHFHWGSLDGQGSEHTVDKKKYAAELHL...
PTM Classifier 5-Fold CV: {'cv_f1': np.float64(0.8688788200444387), 'cv_f1_std': np.float64(0.03818794802680322), 'cv_accuracy': np.float64(0.8923076923076924), 'cv_accuracy_std': np.float64(0.028781979898261072)}
Total plots: 70
Saved: /content/protein_ai_report.html


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>